# ASSIGNMENT 4
# Submission Deadline: April 16, 6PM
# Submission Link: https://forms.gle/G4B6FiAsyoPLCkZu9  

# Table of Contents

1. [Provide Information](#Provide-Information)
2. [Instructions](#Instructions)
3. [Environment](#Environment)
4. [Hyperparameters](#Hyperparameters)
5. [Helper Functions](#helper)
6. [DDPG](#ddpg)
7. [TD3](#td3)
8. [PPO](#ppo)
9. [Experiments to Run](#experiments)

# Provide Information
<a id="Provide-Information"></a>

Name: **Somya Gupta**

Roll No.: **211049**

IITK EMail: **somyavg21@iitk.ac.in**

# Instructions
<a id="Instructions"></a>


**Read all the instructions below carefully before you start working on the assignment.**
- The purpose of this course is that you learn RL and the best way to do that is by implementation and experimentation.
- The assignment requires your to implement some algorithms and you are required report your findings after experimenting with those algorithms.
- **You are required to submit ZIP file containing a Jupyter notebook (.ipynb), and an image folder. The notebook would include the code, graphs/plots of the experiments you run and your findings/observations. Image folder is the folder having plots, images, etc.**
- In case you use any maths in your explanations, render it using latex in the Jupyter notebook.
- You are expected to implement algorithms on your own and not copy it from other sources/class mates. Of course, you can refer to lecture slides.
- If you use any reference or material (including code), please cite the source, else it will be considered plagiarism. But referring to other sources that directly solve the problems given in the assignment is not allowed. There is a limit to which you can refer to outside material.
- This is an individual assignment.
- In case your solution is found to have an overlap with solution by someone else (including external sources), all the parties involved will get zero in this and all future assignments plus further more penalties in the overall grade. We will check not just for lexical but also semantic overlap. Same applies for the code as well. Even an iota of cheating would NOT be tolerated. If you cheat one line or cheat one page the penalty would be same.
- Be a smart agent, think long term, if you cheat we will discover it somehow, the price you would be paying is not worth it.
- In case you are struggling with the assignment, seek help from TAs. Cheating is not an option! I respect honesty and would be lenient if you are not able to solve some questions due to difficulty in understanding. Remember we are there to help you out, seek help if something is difficult to understand.
- The deadline for the submission is given above. Submit at least 30 minutes before the deadline, lot can happen at the last moment, your internet can fail, there can be a power failure, you can be abducted by aliens, etc.
- You have to submit your assignment via the Google Form (link above)
- The form would close after the deadline and we will not accept any solution. No reason what-so-ever would be accepted for not being able to submit before the deadline.
- Since the assignment involves experimentation, reporting your results and observations, there is a lot of scope for creativity and innovation and presenting new perspectives. Such efforts would be highly appreciated and accordingly well rewarded. Be an exploratory agent!
- Your code should be very well documented, there are marks for that.
- In your plots, have a clear legend and clear lines, etc. Of course you would generating the plots in your code but you must also put these plots in your notebook. Generate high resolution pdf/svg version of the plots so that it doesn't pixilate on zooming.
- For all experiments, report about the seed used in the code documentation, write about the seed used.
- In your notebook write about all things that are not obvious from the code e.g., if you have made any assumptions, references/sources, running time, etc.
-  **DO NOT Forget to write name, roll no and email details above**
- **In addition to checking your code, we will be conducting one-on-one viva for the evaluation. So please make sure that you do not cheat!**
- **Use of LLMs based tools or AI-based code tools is strictly prohibited! Use of ChatGPT, VS Code, Gemini, CO-Pilot, etc. is not allowed. NOTE VS code is also not allowed. Even in Colab disable the AI assistant. If you use it, we will know it very easily. Use of any of the tools would be counted as cheating and would be given a ZERO, with no questions asked.**
- For each of the sub-part in the question create a new cell below the question and put your answer in there. This includes the plots as well

# OpenAI Gym Environments
<a id="Environment"></a>

In [1]:
# !pip install gymnasium

In [1]:
# all imports go in here
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import time
import random
from itertools import count, cycle
from collections import deque, namedtuple

In this assignment we will be exploring Deep RL algorithms and for this we will be using environmentd provided by OpenAI Gym. In particular we will be exploring "Pendulum-v1" , "Hopper-v4", and "Half-Cheetah" environments (https://gymnasium.farama.org/environments/classic_control/ ). The code to instantiate the environments are given in the cells below. Run these cells and play with the environments to learn more details about the environments.

In [5]:
# Create Inverted Pendulum environment
#https://gymnasium.farama.org/environments/classic_control/cart_pole/

env = gym.make('Pendulum-v1', render_mode="rgb_array")
s = env.reset(seed = 34)
print("Observation Space = ")
print(env.observation_space)
print("Action Space = ")
print(env.action_space)
done = False
for episode in range(3):
    print("In episode {}".format(episode))
    for i in range(10):
        env.render()
        a = env.action_space.sample()
        s, r, done, truncated, _ = env.step(a)
        print(f"s:{s}, a:{a}")
        if done:
            print("Finished after {} timestep".format(i+1))
env.close()

Observation Space = 
Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
Action Space = 
Box(-2.0, 2.0, (1,), float32)
In episode 0
s:[-0.99720293 -0.07474157  0.9900227 ], a:[1.7643301]
s:[-0.99390554 -0.1102349   0.712961  ], a:[-1.4733703]
s:[-0.98944163 -0.14493208  0.69969857], a:[0.46275806]
s:[-0.9821502  -0.18809825  0.8756232 ], a:[1.8974915]
s:[-0.97705686 -0.21297866  0.50794166], a:[-1.5107193]
s:[-0.9713927  -0.23747897  0.502944  ], a:[1.0315756]
s:[-0.9691915  -0.24630834  0.18199271], a:[-0.95228016]
s:[-0.9724616  -0.23306331 -0.27285665], a:[-1.8007872]
s:[-0.97959375 -0.2009876  -0.65721124], a:[-1.3970473]
s:[-0.98799896 -0.15446046 -0.9456933 ], a:[-0.91827583]
In episode 1
s:[-0.995774   -0.09183768 -1.2622813 ], a:[-1.3382845]
s:[-0.9998935  -0.01459602 -1.5474147 ], a:[-1.4417005]
s:[-0.9981234   0.06123441 -1.5173856 ], a:[0.27317357]
s:[-0.9906399   0.13650127 -1.5131202 ], a:[-0.2777356]
s:[-0.98123246  0.19282845 -1.1423025 ], a:[1.7896111]
s:[-0.9718689   0.23552

In [7]:
# !pip install swig
# !pip install gymnasium[box2d]
# !pip install gymnasium[mujoco]

# Create Hopper environment
# https://gymnasium.farama.org/environments/mujoco/hopper/


import gymnasium as gym
env = gym.make("Hopper-v4", render_mode = "rgb_array")
s = env.reset(seed = 34)
print("Observation Space = ")
print(env.observation_space)
print("Action Space = ")
print(env.action_space)
done = False
for episode in range(1):
    print("In episode {}".format(episode))
    for i in range(10):
        # env.render()
        a = env.action_space.sample()
        s, r, done, truncated, _ = env.step(a)
        print(f"s:{s}, a:{a}")
        if done:
            print("Finished after {} timestep".format(i+1))
env.close()


Observation Space = 
Box(-inf, inf, (11,), float64)
Action Space = 
Box(-1.0, 1.0, (3,), float32)
In episode 0
s:[ 1.25357179e+00 -5.81077165e-03  3.15209995e-04 -4.57306166e-03
 -8.68540545e-04 -1.35934785e-01 -4.20324799e-02 -8.09156054e-01
 -3.03224225e-01 -1.10645750e+00 -9.39301897e-01], a:[-0.29215994 -0.89188826 -0.7051129 ]
s:[ 1.25280733e+00 -9.84325810e-03  8.41072617e-04 -1.39076905e-02
 -3.42535638e-03 -8.79986910e-02 -1.48705502e-01 -3.33736217e-01
  2.67826572e-01 -1.21317901e+00  3.00544935e-01], a:[ 0.9037323  -0.00126938  0.8645338 ]
s:[ 1.25129189e+00 -1.39841905e-02  2.32571910e-03 -2.56500403e-02
  4.16385396e-05 -1.37406248e-01 -2.30645307e-01 -6.89196496e-01
  1.18195362e-01 -1.72309103e+00  5.65572443e-01], a:[ 0.4126885  -0.38827196  0.16556203]
s:[ 1.24905539 -0.02161211  0.00302328 -0.04381936  0.0086596  -0.21919277
 -0.32997315 -1.21290272  0.06141689 -2.81843942  1.58718608], a:[ 0.85929924 -0.79538786  0.6749351 ]
s:[ 1.24609303e+00 -3.28421466e-02 -1.1830

In [9]:

# !pip install swig
# !pip install gymnasium[box2d]

# Create Half-Cheetah environment
# https://gymnasium.farama.org/environments/mujoco/hopper/


import gymnasium as gym
env = gym.make("HalfCheetah-v4", render_mode = "rgb_array")
s = env.reset(seed = 34)
print("Observation Space = ")
print(env.observation_space)
print("Action Space = ")
print(env.action_space)
done = False
for episode in range(1):
    print("In episode {}".format(episode))
    for i in range(10):
        # env.render()
        a = env.action_space.sample()
        s, r, done, truncated, _ = env.step(a)
        print(f"s:{s}, a:{a}")
        if done:
            print("Finished after {} timestep".format(i+1))
env.close()


Observation Space = 
Box(-inf, inf, (17,), float64)
Action Space = 
Box(-1.0, 1.0, (6,), float32)
In episode 0
s:[ 5.51784510e-02 -2.56528508e-02 -2.00737503e-01  3.09103873e-02
 -1.21102797e-01  1.12659708e-01 -1.06203521e-01  3.13405082e-02
 -5.62795996e-01 -6.33737134e-01  6.82852022e-01 -6.25015323e+00
  2.58348249e-03 -4.45592220e+00  9.01496399e-01 -4.49023442e+00
  6.56435286e-01], a:[-0.81847113 -0.11929753 -0.44011053  0.23883124 -0.2803796   0.09812822]
s:[ 2.64331864e-03 -3.44568187e-02 -1.71140473e-01  1.91294295e-01
  1.90320904e-01  4.35302242e-01 -4.72859355e-01  1.12534823e-01
  7.81173659e-01 -1.25465951e+00 -6.60182853e-01  4.90384510e+00
  4.23573637e+00  9.74108814e+00  7.85296165e+00 -6.29151529e+00
  1.86803630e+00], a:[ 0.5865437   0.9123208   0.9489244   0.7900563  -0.9769559   0.29271504]
s:[-3.97709668e-02 -6.65875072e-02  2.16521330e-01  4.75207897e-03
 -1.41849809e-01  3.67782364e-01  8.67040299e-02 -5.34206955e-02
  3.01137855e-01 -8.81837530e-01 -5.2707580

# Hyperparameters
<a id="Hyperparameters"></a>

All your hyperparameters should be stated here. We will change their value here and your code should work  accordingly.

In [10]:

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Helper Functions
<a id="helper"></a>

Write all the helper functions that will be used for value-based and policy based algorithms below. In case you want to add more helper functions, please feel free to add.

In [79]:
#Value Network
def createValueNetwork(inDim, outDim, action_size, hDim = [32,32], activation = F.relu):
    #this creates a Feed Forward Neural Network class and instantiates it and returns the class
    #the class should be derived from torch nn.Module and it should have init and forward method at the very least
    #the forward function should return q-value for each possible action

    #Your code goes in here
    class criticNetwork(nn.Module):
      def __init__(self,input_dim,output_dim,n_action,hidden_dim,activation_fn):
        super(criticNetwork,self).__init__()
        self.activation_fn=activation_fn
        self.hidden_dim=hidden_dim
        self.hidden_dim[0]=self.hidden_dim[0]+n_action

        self.fc1 = nn.Linear(input_dim,hidden_dim[0])
        self.fc2 = nn.Linear(hidden_dim[0]+n_action, hidden_dim[1])
        self.fc3 = nn.Linear(hidden_dim[1], 1)
        self.relu = nn.ReLU()

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

      def forward(self,state,action):
        x=state
        a=action
        out = self.fc1(x)
        out = self.relu(out)

        out = self.fc2(torch.cat([out,a],1))
        out = self.relu(out)
        out = self.fc3(out)
        return out

    return criticNetwork(inDim, outDim, action_size, hDim, activation)

In [80]:
#Policy Network
def createPolicyNetwork(inDim, outDim, hDim = [32,32], activation = F.relu, output_activation_fn=torch.tanh):
    #this creates a Feed Forward Neural Network class and instantiates it and returns the class
    #the class should be derived from torch nn.Module and it should have init and forward method at the very least
    #the forward function should return action logit vector
    #Your code goes in here
    class actorNetwork(nn.Module):
      def __init__(self,input_dim,output_dim,hidden_dim,activation_fn,output_activation_fn):
        super(actorNetwork,self).__init__()
        self.activation_fn=activation_fn
        self.output_activation_fn=output_activation_fn

        self.fc1 = nn.Linear(input_dim, hidden_dim[0])
        self.fc2 = nn.Linear( hidden_dim[0],  hidden_dim[1])
        self.fc3 = nn.Linear( hidden_dim[1], output_dim)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

      def forward(self,state):
        out = self.fc1(state)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.tanh(out)
        return out

    return actorNetwork(inDim, outDim, hDim, activation, output_activation_fn)

## ReplayBuffer

In next few cells, you will implement replaybuffer class.

This class creates a buffer for storing and retrieving experiences. This is a generic class and can be used
for different agents like NFQ, DQN, DDQN, PER_DDQN, etc.
Following are the methods for this class which are implemented in subsequent cells

```
class ReplayBuffer():
    def __init__(self, bufferSize, batch_size, seed)
    def store(self, state, action, reward, next_state, done)
    def sample(self, batchSize)
    def length(self)
```   

In [81]:
class ReplayBuffer():
    def __init__(self, buffer_size):
        # this function creates the relevant data-structures, and intializes all relevant variables
        #Your code goes in here

        self.ss_mem = np.empty(shape=(buffer_size), dtype=np.ndarray)
        self.as_mem = np.empty(shape=(buffer_size), dtype=np.ndarray)
        self.rs_mem = np.empty(shape=(buffer_size), dtype=float)
        self.ps_mem = np.empty(shape=(buffer_size), dtype=np.ndarray)
        self.ds_mem = np.empty(shape=(buffer_size), dtype=bool)

        self.buffer_size = buffer_size
        self._idx = 0
        self.size = 0
        # Assume default batch size for sampling if not provided
        self.default_batch_size = 64

        return

In [82]:
class ReplayBuffer(ReplayBuffer):
    def store(self, experience):
        #stores the experiences, based on parameters in init
        #
        #this function does not return anything
        #
        #Your code goes in here
        s, a, r, p, d = experience
        self.ss_mem[self._idx] = s
        self.as_mem[self._idx] = a
        self.rs_mem[self._idx] = r
        self.ps_mem[self._idx] = p
        self.ds_mem[self._idx] = d

        self._idx = (self._idx + 1) % self.buffer_size
        self.size = min(self.size + 1, self.buffer_size)

        return

In [83]:
class ReplayBuffer(ReplayBuffer):
    def sample(self, batch_size):
        # this method returns batchSize number of experiences
        # this function returns experiences samples
        #
        #Your code goes in here
        idxs = np.random.choice(self.size, batch_size, replace=False)
        experiences = (np.vstack(self.ss_mem[idxs]), \
                        np.vstack(self.as_mem[idxs]), \
                        np.vstack(self.rs_mem[idxs]), \
                        np.vstack(self.ps_mem[idxs]), \
                        np.vstack(self.ds_mem[idxs]))

        return experiences

In [84]:
class ReplayBuffer(ReplayBuffer):
  def splitExperiences(self, experiences):
        states, actions, rewards, nextStates, dones = experiences

        return states, actions, rewards, nextStates, dones

In [85]:
class ReplayBuffer(ReplayBuffer):
    def length(self):
        #tells the number of experiences stored in the internal buffer
        #
        #Your code goes in here
        return self.size

## Deep Deterministic Policy Gradient (DDPG) ##
<a id="ddpg"></a>

Implement the Deep Deterministic Policy Gradient (DDPG) agent. We have studied about DDPG agent in the Lecture. Use the function definitions (given below).

This class implements the DDPG agent, you are required to implement the various methods of this class
as outlined below. Note this class is generic and should work with any permissible Gym environment

```
class DDPG():
    def init(self, env, seed, gamma, tau, bufferSize, batch_size, updateFrequency,
             policyOptimizerFn, valueOptimizerFn,
             policyOptimizerLR,valueOptimizerLR,
             MAX_TRAIN_EPISODES,MAX_EVAL_EPISODE,
             optimizerFn)
    
    def runDDPG(self)
    def trainAgent(self)
    def gaussianStrategy(self, net , s , envActionRange , noiseScaleRatio,
        explorationMax = True)
    def greedyStrategy(self, net , s , envActionRange)
    def trainNetworks(self, experiences)
    def updateNetworks(self, onlineNet, targetNet, tau)
    def evaluateAgent(self)




```

In [86]:
class DDPG():
    def __init__(self,env_id,seed,gamma,tau,buffer_size,batch_size,update_freq,policyOptimizer,valueOptimizer,policyOptimizerLr,valueOptimizerLr,max_train_eps,max_eval_eps):
      self.env = gym.make(env_id)
      self.seed = seed
      self.env.reset(seed=self.seed)

      torch.manual_seed(seed)
      np.random.seed(seed)
      random.seed(seed)

      self.gamma=gamma
      self.tau=tau
      self.update_freq=update_freq
      self.max_train_eps=max_train_eps
      self.max_eval_eps=max_eval_eps
      self.batch_size=batch_size

      self.n_action=self.env.action_space.shape[0]
      self.n_state=self.env.observation_space.shape[0]
      self.actionLowValue=self.env.action_space.low
      self.actionHighValue=self.env.action_space.high
      self.actionRange=(self.actionLowValue,self.actionHighValue)

      self.rBuffer=ReplayBuffer(buffer_size)

      self.target_value_net=createValueNetwork(self.n_state,1,self.n_action,[32,32],F.relu)
      self.online_value_net=createValueNetwork(self.n_state,1,self.n_action,[32,32],F.relu)
      self.target_policy_net=createPolicyNetwork(self.n_state,self.n_action,[32,32],F.relu,torch.tanh)
      self.online_policy_net=createPolicyNetwork(self.n_state,self.n_action,[32,32],F.relu,torch.tanh)

      self.optimizerP = policyOptimizer(self.online_policy_net.parameters(), policyOptimizerLr)
      self.optimizerV = valueOptimizer(self.online_value_net.parameters(), valueOptimizerLr)


      self.updateNetworks(self.online_value_net,self.target_value_net, self.tau)
      self.updateNetworks(self.online_policy_net,self.target_policy_net, self.tau)

    def initBookeeping(self):
      self.episode_step = []
      self.episode_reward = []
      self.training_time = []
      self.evaluation_scores = []
      # self.wallClock_time=[]

    def performBookeeping(self,train=True):
      if train:
        self.episode_step.append(self.step)
        self.episode_reward.append(self.Tr/1000)
        self.training_time.append(self.episode_elapsed)
        self.evaluation_scores.append(self.evalscore)



        #Your code goes in here




In [87]:
class DDPG(DDPG):
    def updateNetworks(self, onlineNet, targetNet, tau):
        #this function updates the onlineNetwork with the target network
        #
        # Your code goes in here
        for target, online in zip(targetNet.parameters(),
                                  onlineNet.parameters()):
            target.data.copy_(tau*online.data+(1-tau)*target.data)




In [88]:
class DDPG(DDPG):
    def gaussianStrategy (self, net , s , envActionRange , noiseScaleRatio ,
        explorationMax = True ):
        #this function sets the scale of exploration then add the noise of this scale to the greedy action
        #and clips it within the range

        #Your code here
        actionLowValue, actionHighValue=envActionRange
        if explorationMax:
          scale=actionHighValue
        else:
          scale=noiseScaleRatio*actionHighValue

        greedyAction=net(torch.from_numpy(s)).detach()
        noise=np.random.normal(0, scale, len(actionHighValue))
        action=greedyAction+noise
        action=np.clip(action,actionLowValue,actionHighValue)

        return action


In [89]:
class DDPG(DDPG):
    def greedyStrategy(self,net,s,actionRange):
      actionLowValue,actionHighValue=actionRange
      action=net(torch.from_numpy(s)).detach()
      action=np.clip(action,actionLowValue,actionHighValue)
      return action


In [90]:
class DDPG(DDPG):
    def runDDPG (self):
        #this is the main method, it trains the agent, performs bookkeeping while training and finally evaluates
        #the agent and returns the following quantities:
        #1. episode wise mean train rewards
        #2. epsiode wise mean eval rewards
        #2. episode wise trainTime (in seconds): time elapsed during training since the start of the first episode
        #3. episode wise wallClockTime (in seconds): actual time elapsed since the start of training,
        #                               note this will include time for BookKeeping and evaluation
        # Note both trainTime and wallClockTime get accumulated as episodes proceed.
        #
        #Your code goes in here
        resultTrain=self.trainAgent()
        steps,rewards,evalScore,trainingTime=resultTrain
        resultEval=self.evaluateAgent()


        return rewards, trainingTime, evalScore, resultEval


In [129]:
class DDPG(DDPG):
    def trainAgent(self):
        #this method collects experiences and trains the agent and does BookKeeping while training.
        #this calls the trainNetwork() method internally, it also evaluates the agent per episode
        #it trains the agent for MAX_TRAIN_EPISODES
        #
        #Your code goes in here
        self.updateNetworks(self.online_value_net,self.target_value_net, self.tau)
        self.updateNetworks(self.online_policy_net,self.target_policy_net, self.tau)
        self.initBookeeping()
        episode_start=time.time()
        for e in range(self.max_train_eps):
          s,done=self.env.reset(seed=self.seed)
          print(e)
          self.step=0
          self.Tr=0
          self.evalscore=0

          for i in range(1000):
            explorationMax=self.rBuffer.length()<400
            a=self.gaussianStrategy(self.online_policy_net, s, self.actionRange, 0.5, explorationMax)
            s_n, r, done, _, _ =self.env.step(a)
            experience=(s,a,r,s_n,done)
            self.rBuffer.store(experience)


            if self.rBuffer.length()>300:
              experiences=self.rBuffer.sample(self.batch_size)
              self.trainNetwork(experiences)

            if e%self.update_freq==0:
              self.updateNetworks(self.online_value_net,self.target_value_net, self.tau)
              self.updateNetworks(self.online_policy_net,self.target_policy_net, self.tau)

            s=s_n

            self.step+=1
            self.Tr+=r

            if done:
              break

          self.evalscore,_=self.evaluateAgent()
          self.episode_elapsed = time.time() - episode_start
          self.performBookeeping(train=True)

        return self.episode_step, self.episode_reward, self.evaluation_scores, self.training_time



In [130]:
class DDPG(DDPG):
    def trainNetwork(self, experiences):
        # this method trains the value network epoch number of times and is called by the trainAgent function
        # it essentially uses the experiences to calculate target, using the targets it calculates the error, which
        # is further used for calulating the loss. It then uses the optimizer over the loss
        # to update the params of the network by backpropagating through the network
        # this function does not return anything
        # you can try out other loss functions other than MSE like Huber loss, MAE, etc.
        #
        #Your code goes in here
        s, a, r, s_n, done=self.rBuffer.splitExperiences(experiences)
        argmax_a_qs_v=self.target_policy_net(torch.from_numpy(s_n)).detach()
        max_a_qs_v=self.target_value_net(torch.from_numpy(s_n), argmax_a_qs_v).detach()

        max_a_qs_v=max_a_qs_v.numpy()
        target_qs=r+self.gamma*max_a_qs_v*((1-done).T)
        a=a.astype(np.float32)
        qs=self.online_value_net(torch.from_numpy(s),torch.from_numpy(a))
        tdError=torch.from_numpy(target_qs)-qs
        valueLoss=tdError.pow(2).mul(0.5).mean()
        self.optimizerV.zero_grad()
        valueLoss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_value_net.parameters(), max_norm=15)
        self.optimizerV.step()
        argmax_a_qs_p=self.online_policy_net(torch.from_numpy(s_n)).detach()
        max_a_qs_p=self.online_value_net(torch.from_numpy(s_n),argmax_a_qs_p)
        policyLoss=-1.0*torch.mean(max_a_qs_p)
        self.optimizerP.zero_grad()
        policyLoss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_value_net.parameters(), max_norm=15)
        self.optimizerP.step()


In [131]:
class DDPG(DDPG):
    def evaluateAgent(self):
        #this function evaluates the agent using the value network, it evaluates agent for MAX_EVAL_EPISODES
        #typcially MAX_EVAL_EPISODES = 1
        #
        #Your code goes in here
        rewards=[]
        for ep in range(self.max_eval_eps):
          rs=0
          s,done=self.env.reset(seed=self.seed)
          for c in range(500):
            a=self.greedyStrategy(self.online_policy_net,s,self.actionRange)

            s,r,done, _, _=self.env.step(a)
            rs+=r
            if done:
              break
          rewards.append(rs)

        return np.mean(rewards), np.std(rewards)

In [132]:

# seed_list = [420,133,74,317,233]
# for seed in seed_list:
gamma=0.99
tau=0.1
buffer_size=5000
update_freq=10
policyOptimizer=torch.optim.Adam
valueOptimizer=torch.optim.Adam
policyOptimizerLr=3*1e-3
valueOptimizerLr=3*1e-3
max_train_eps=120
max_eval_eps=1
batch_size=224


In [133]:
def plotQuantity(eps,env_id):
  seeds=[55,45]
  # Steps=[]
  Rewards=[]
  EvalScore=[]
  TrainingTime=[]
  ResultEval=[]

  for s in seeds:

    agent=DDPG(env_id,s,gamma,tau,buffer_size,batch_size, update_freq,policyOptimizer,valueOptimizer,policyOptimizerLr,valueOptimizerLr,max_train_eps,max_eval_eps)
    rewards,evalScore,trainingTime,resultEval=agent.runDDPG()
    # Steps.append(steps)
    Rewards.append(rewards)
    EvalScore.append(evalScore)
    TrainingTime.append(trainingTime)
    ResultEval.append(resultEval)
    print(ResultEval)
  quantities = [Rewards, EvalScore, TrainingTime]
  quantity_names = ["Rewards", "EvalScore", "TrainingTime"]

  for quantity, quantity_name in zip(quantities, quantity_names):
      mean_values = np.mean(quantity, axis=0)
      min_values = np.min(quantity, axis=0)
      max_values = np.max(quantity, axis=0)

    # Calculate mean, min, and max values across environment instances for each episode

    # for episode in range(eps):
    #     quantity_values = quantityList[episode]

    #     mean_values.append(np.mean(quantity_values))
    #     min_values.append(np.min(quantity_values))
    #     max_values.append(np.max(quantity_values))

    # Create the plot
      plt.figure(figsize=(10, 6))

      # Plot mean values
      plt.plot(range(1, eps + 1), mean_values, label='Mean', color='blue')

      # Fill between min and max values
      plt.fill_between(range(1, eps + 1), min_values, max_values, color='lightblue', alpha=0.3)

      # Add legend
      plt.legend(loc='upper left')

      # Add labels and title
      plt.xlabel('Episode')
      plt.ylabel(quantity_name)
      plt.title(env_id)

      # Show plot
      plt.show()

In [134]:
envs=["Pendulum-v1","Hopper-v4","HalfCheetah-v4"]
for env in envs:
  plotQuantity(120,env)

0


/usr/local/lib/python3.10/dist-packages/gymnasium/utils/passive_env_checker.py:246: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'torch.Tensor'>
  logger.warn(


1
2
3
4
5
6
7
8
9
10
11
12
13
14


KeyboardInterrupt: 

# Twin-Delayed Deep Deterministic Policy Gradient (TD3)
<a id="td3"></a>

Implement the Twin-delayed deep deterministic policy gradient (TD3) agent. We have studied about TD3 agent in the Lecture. Use the function definitions (given below).

This class implements the TD3 agent, you are required to implement the various methods of this class
as outlined below. Note this class is generic and should work with any permissible Gym environment

```
class DDPG():
    def init(env, gamma, tau,
    bufferSize ,
    updateFrequencyPolicy ,
    updateFrequencyValue ,
    trainPolicyFrequency ,
    policyOptimizerFn ,
    valueOptimizerFn ,
    policyOptimizerLR ,
    valueOptimizerLR ,
    MAX TRAIN EPISODES,
    MAX EVAL EPISODE,
    optimizerFn )
    
    def runTD3 (self)
    def trainAgent (self)
    def gaussianStrategy (self, net , s , envActionRange , noiseScaleRatio ,
        explorationMax = True)
    def greedyStrategy (self, net , s , envActionRange)
    def trainNetworks (self,experiences , envActionRange)
    def updateValueNetwork(self, onlineNet, targetNet, tau)
    def updatePolicyNetwork(self, onlineNet, targetNet, tau)
    def evaluateAgent (self)



```

In [135]:
class TD3():
    # def __init__(self,env, gamma, tau,
    # bufferSize ,
    # update_freq_policy ,
    # update_freq_value ,
    # trainPolicyFrequency ,
    # policyOptimizerFn ,
    # valueOptimizerFn ,
    # policyOptimizerLR ,
    # valueOptimizerLR ,
    # MAX_TRAIN_EPISODES,
    # MAX_EVAL_EPISODE,
    # optimizerFn):
        #this TD3 method
        # 1. creates and initializes (with seed) the environment, train/eval episodes, gamma, etc.
        # 2. creates and intializes all the variables required for book-keeping values via the initBookKeeping method
        # 3. creates targetValueNetwork , targetPolicyNetwork
        # 4. creates and initializes (with network params) the optimizer function
        # 5. creates onlineValueNetwork, onlinePolicyNetwork
        # 6. Creates the replayBuffer

        #Your code goes in here
    def __init__(self,env_id,seed,gamma,tau,buffer_size,batch_size,update_freq_policy, update_freq_value, trainPolicyFrequency, policyOptimizer,valueOptimizer,policyOptimizerLr,valueOptimizerLr,max_train_eps,max_eval_eps):
      self.env = gym.make(env_id)
      self.seed = seed
      self.env.reset(seed=self.seed)

      torch.manual_seed(seed)
      np.random.seed(seed)
      random.seed(seed)

      self.gamma=gamma
      self.tau=tau
      self.update_freq_policy=update_freq_policy
      self.update_freq_value=update_freq_value
      self.max_train_eps=max_train_eps
      self.max_eval_eps=max_eval_eps
      self.batch_size=batch_size
      self.trainPolicyFrequency=trainPolicyFrequency

      self.n_action=self.env.action_space.shape[0]
      self.n_state=self.env.observation_space.shape[0]
      self.actionLowValue=self.env.action_space.low
      self.actionHighValue=self.env.action_space.high
      self.actionRange=(self.actionLowValue,self.actionHighValue)

      self.rBuffer=ReplayBuffer(buffer_size)

      self.target_value_net=createValueNetwork(self.n_state,1,self.n_action,[32,32],F.relu)
      self.online_value_net=createValueNetwork(self.n_state,1,self.n_action,[32,32],F.relu)
      self.target_policy_net=createPolicyNetwork(self.n_state,self.n_action,[32,32],F.relu,torch.tanh)
      self.online_policy_net=createPolicyNetwork(self.n_state,self.n_action,[32,32],F.relu,torch.tanh)

      self.optimizerP = policyOptimizer(self.online_policy_net.parameters(), policyOptimizerLr)
      self.optimizerV = valueOptimizer(self.online_value_net.parameters(), valueOptimizerLr)




    def initBookeeping(self):
      self.episode_step = []
      self.episode_reward = []
      self.training_time = []
      self.evaluation_scores = []
      # self.wallClock_time=[]

    def performBookeeping(self,train=True):
      if train:
        self.episode_step.append(self.step)
        self.episode_reward.append(self.Tr/1000)
        self.training_time.append(self.episode_elapsed)
        self.evaluation_scores.append(self.evalscore)





In [136]:
class TD3(TD3):
    def updateValueNetwork(self, tau):
        #this function updates the onlineValueNetwork with the targetValuenetwork
        #
        # Your code goes in here
        for target, online in zip(self.target_value_net.parameters(),
                                  self.online_value_net.parameters()):
            target.data.copy_(tau*online.data+(1-tau)*target.data)



In [137]:
class TD3(TD3):
    def updatePolicyNetwork(self, tau):
        #this function updates the onlinePolicuNetwork with the targetPolicynetwork
        #
        # Your code goes in here
        for target, online in zip(self.target_policy_net.parameters(),
                                  self.target_policy_net.parameters()):
            target.data.copy_(tau*online.data+(1-tau)*target.data)


In [138]:
class TD3(TD3):
    def gaussianStrategy (self, net , s , envActionRange , noiseScaleRatio ,
        explorationMax = True ):
        #this function sets the scale of exploration then add the noise of this scale to the greedy action
        #and clips it within the range

        #Your code here
        actionLowValue, actionHighValue=envActionRange
        if explorationMax:
          scale=actionHighValue
        else:
          scale=noiseScaleRatio*actionHighValue

        greedyAction=net(torch.from_numpy(s)).detach()
        noise=np.random.normal(0, scale, len(actionHighValue))
        action=greedyAction+noise
        action=np.clip(action,actionLowValue,actionHighValue)

        return action


In [139]:
class TD3(TD3):
    def greedyStrategy (self, net , s , actionRange ):
        #this function selects the greedy action
        #and clips it within the range

        #Your code here
        actionLowValue,actionHighValue=actionRange
        action=net(torch.from_numpy(s)).detach()
        action=np.clip(action,actionLowValue,actionHighValue)
        return action


In [140]:
class TD3(TD3):
    def runTD3 (self):
        #this is the main method, it trains the agent, performs bookkeeping while training and finally evaluates
        #the agent and returns the following quantities:
        #1. episode wise mean train rewards
        #2. epsiode wise mean eval rewards
        #2. episode wise trainTime (in seconds): time elapsed during training since the start of the first episode
        #3. episode wise wallClockTime (in seconds): actual time elapsed since the start of training,
        #                               note this will include time for BookKeeping and evaluation
        # Note both trainTime and wallClockTime get accumulated as episodes proceed.
        #
        #Your code goes in here
        #
        resultTrain=self.trainAgent()
        steps,rewards,evalScore,trainingTime=resultTrain
        resultEval=self.evaluateAgent()


        return rewards, trainingTime, evalScore, resultEval


In [145]:
class TD3(TD3):
    def trainAgent(self):
        #this method collects experiences and trains the agent and does BookKeeping while training.
        #this calls the trainNetwork() method internally, it also evaluates the agent per episode
        #it trains the agent for MAX_TRAIN_EPISODES
        #
        #Your code goes in here
        self.updateValueNetwork(self.tau)
        self.updatePolicyNetwork(self.tau)
        self.initBookeeping()
        for e in range(self.max_train_eps):
          s,done=self.env.reset(seed=self.seed)
          self.step=0
          self.Tr=0
          self.evalscore=0
          episode_start=time.time()
          for i in range(1000):
            explorationMax=self.rBuffer.length()<400
            a=self.gaussianStrategy(self.online_policy_net, s, self.actionRange, 0.5, explorationMax)
            s_n, r, done, _, _ =self.env.step(a)
            experience=(s,a,r,s_n,done)
            self.rBuffer.store(experience)


            if self.rBuffer.length()>300:
              experiences=self.rBuffer.sample(self.batch_size)
              self.trainNetwork(experiences, e)

            if e%self.update_freq_policy==0:
              self.updatePolicyNetwork(self.tau)

            if e%self.update_freq_value==0:
              self.updateValueNetwork(self.tau)

            s=s_n

            self.step+=1
            self.Tr+=r

            if done:
              break

          self.evalscore,_=self.evaluateAgent()
          self.episode_elapsed = time.time() - episode_start
          self.performBookeeping(train=True)

        return self.episode_step, self.episode_reward, self.evaluation_scores, self.training_time


In [146]:
class TD3(TD3):
    def trainNetwork(self,experiences , episode):
        # this method trains the value network epoch number of times and is called by the trainAgent function
        # it essentially uses the experiences to calculate target, using the targets it calculates the error, which
        # is further used for calulating the loss. It then uses the optimizer over the loss
        # to update the params of the network by backpropagating through the network
        # this function does not return anything
        # you can try out other loss functions other than MSE like Huber loss, MAE, etc.
        #
        #Your code goes in here
        #self.actionRange=(self.actionLowValue,self.actionHighValue)
        s, a, r, s_n, done=self.rBuffer.splitExperiences(experiences)
        a_noise=(self.actionHighValue-self.actionLowValue)*(torch.randn_like(a))
        a_noise=torch.max(torch.min(a_noise,self.actionHighValue),self.actionLowValue)

        argmax_a_qs_v=self.target_policy_net(torch.from_numpy(s_n))
        noisy_argmax_a_qs_v=argmax_a_qs_v+a_noise
        noisy_argmax_a_qs_v=torch.max(torch.min(noisy_argmax_a_qs_v,self.actionHighValue),self.actionLowValue)
        max_1_a_qs_v, max_2_a_qs_v=self.target_value_net(torch.from_numpy(s_n), noisy_argmax_a_qs_v)

        max_a_qs_v=min(max_1_a_qs_v,max_2_a_qs_v)
        max_a_qs_v=max_a_qs_v.numpy()
        target_qs=r+self.gamma*max_a_qs_v*((1-done).T)
        a=a.astype(np.float32)
        qs_1, qs_2=self.online_value_net(torch.from_numpy(s),torch.from_numpy(a))
        tdError_1=torch.from_numpy(target_qs).detach()-qs_1
        tdError_2=torch.from_numpy(target_qs).detach()-qs_2

        valueLoss=torch.mean(0.5*(tdError_1)**2)+torch.mean(0.5*(tdError_2)**2)
        self.optimizerV.zero_grad()
        valueLoss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_value_net.parameters(), max_norm=15)
        self.optimizerV.step()

        if episode%self.trainPolicyFrequency==0:
          argmax_a_qs_p=self.online_policy_net(torch.from_numpy(s_n))
          max_a_qs_p=self.online_value_net(torch.from_numpy(s_n),argmax_a_qs_p)
          policyLoss=-1.0*torch.mean(max_a_qs_p)
          self.optimizerP.zero_grad()
          policyLoss.backward()
          torch.nn.utils.clip_grad_norm_(self.online_value_net.parameters(), max_norm=15)
          self.optimizerP.step()



In [147]:
class TD3(TD3):
    def evaluateAgent(self):
        #this function evaluates the agent using the value network, it evaluates agent for MAX_EVAL_EPISODES
        #typcially MAX_EVAL_EPISODES = 1
        #
        #Your code goes in here
        rewards=[]
        for ep in range(self.max_eval_eps):
          s,done=self.env.reset(seed=self.seed)
          for c in range(500):
            a=self.greedyStrategy(self.online_policy_net,s,self.actionRange)
            s,r,done, _, _=self.env.step(a)
            if done:
              break
          rewards.append(r)

        return np.mean(rewards), np.std(rewards)

        pass

In [ ]:
def plotQuantity(eps,env_id):
  seeds=[55,45]
  # Steps=[]
  Rewards=[]
  EvalScore=[]
  TrainingTime=[]
  ResultEval=[]

  for s in seeds:

    agent=TD3(env_id,seed,gamma,tau,buffer_size,batch_size,ufp, ufv , tpf, policyOptimizer,valueOptimizer,policyOptimizerLr,valueOptimizerLr,max_train_eps,max_eval_eps)
    rewards,evalScore,trainingTime,resultEval=agent.runTD3()
    # Steps.append(steps)
    Rewards.append(rewards)
    EvalScore.append(evalScore)
    TrainingTime.append(trainingTime)
    ResultEval.append(resultEval)
    print(ResultEval)
  quantities = [Rewards, EvalScore, TrainingTime]
  quantity_names = ["Rewards", "EvalScore", "TrainingTime"]

  for quantity, quantity_name in zip(quantities, quantity_names):
      mean_values = np.mean(quantity, axis=0)
      min_values = np.min(quantity, axis=0)
      max_values = np.max(quantity, axis=0)

    # Calculate mean, min, and max values across environment instances for each episode

    # for episode in range(eps):
    #     quantity_values = quantityList[episode]

    #     mean_values.append(np.mean(quantity_values))
    #     min_values.append(np.min(quantity_values))
    #     max_values.append(np.max(quantity_values))

    # Create the plot
      plt.figure(figsize=(10, 6))

      # Plot mean values
      plt.plot(range(1, eps + 1), mean_values, label='Mean', color='blue')

      # Fill between min and max values
      plt.fill_between(range(1, eps + 1), min_values, max_values, color='lightblue', alpha=0.3)

      # Add legend
      plt.legend(loc='upper left')

      # Add labels and title
      plt.xlabel('Episode')
      plt.ylabel(quantity_name)
      plt.title(env_id)

      # Show plot
      plt.show()

In [148]:
ufp=10
ufv=9
tpf=2
envs=["Pendulum-v1","Hopper-v4","HalfCheetah-v4"]
for env in envs:
  plotQuantity(120,env)

# PPO
<a id="PPO"></a>

PPO have quite a few key implementation details.
Please Refer:
"Proximal Policy Optimization Algorithms" [PPO](https://arxiv.org/abs/1707.06347) and
"Implementation Matters in Deep RL: A Case Study on PPO and TRPO" [Implementation Matters](https://openreview.net/forum?id=r1etN1rtPB)

Lets finish things off with an easy implementation of PPO!
A easy way to check you implementation details is running your implementation on some easier environment first and make sure it converges. Like "CartPole-v1" should converge to episodic return of 500 in around 300k steps.

In [ ]:
!pip install gymnasium

In [65]:
#All imports here
## Feel free to add or remove

import os
import random
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical

In [66]:
#Hyperparameters
gym_id = "CartPole-v1"  #The id of the gym environment
learning_rate = 2.5e-4
seed = 1
total_timesteps = 2000 #The total timesteps of the experiments
torch_deterministic = True   #If toggled, `torch.backends.cudnn.deterministic=False
cuda = True

num_envs = 4  #The number of parallel game environments (Yes PPO works with vectorized environments)
num_steps = 128 #The number of steps to run in each environment per policy rollout
anneal_lr = True #Toggle learning rate annealing for policy and value networks
gae = True #Use GAE for advantage computation
gamma =0.99
gae_lambda = 0.95 #The lambda for the general advantage estimation
num_minibatches = 4
update_epochs =4  #The K epochs to update the policy
norm_adv = True  #Toggles advantages normalization
clip_coef = 0.2 #The surrogate clipping coefficient (See what is recommended in the paper!)
clip_vloss = True #Toggles whether or not to use a clipped loss for the value function, as per the paper
ent_coef =0.01  #Coefficient of the entropy
vf_coef = 0.5 #Coefficient of the value function
max_grad_norm = 0.5
target_kl = None #The target KL divergence threshold


batch_size = int(num_envs * num_steps)
minibatch_size = int(batch_size // num_minibatches)


In [67]:
#PPO works with vectorized enviromnets lets make a function that returns a function that returns an environment.
#Refer how to make vectorized environments in gymnasium
def make_env(gym_id, seed, idx):
    def thunk():
        env = gym.make(gym_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.reset(seed=seed)
        return env

    return thunk


In [68]:
#We initialize the layers in PPO , refer paper.
#Lets initialize the layers with this function
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    #Initializes the weights and bias of the layers
    #Your code here
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)

    return layer

In [69]:
#Lets make the Main agent class
class Agent(nn.Module):
    def __init__(self, envs):
        super(Agent, self).__init__()

        state_dim=envs.single_observation_space.shape[0]
        action_dim=envs.single_action_space.n
        h_layers=[64,64]

        self.critic = nn.Sequential(
            layer_init(nn.Linear(state_dim,h_layers[0])),
            nn.Tanh(),
            layer_init(nn.Linear(h_layers[0],h_layers[1])),
            nn.Tanh(),
            layer_init(nn.Linear(h_layers[1],1), std=0.01),
        )

        self.actor = nn.Sequential(
            layer_init(nn.Linear(state_dim,h_layers[0])),
            nn.Tanh(),
            layer_init(nn.Linear(h_layers[0],h_layers[1])),
            nn.Tanh(),
            layer_init(nn.Linear(h_layers[1],action_dim), std=0.01),


        )

    def get_value(self, x):
            # Returns the value from the critic on the observation x
            value=self.critic(x)
            return value

    def get_action_and_value(self, x, action=None):
        #Returns 1.the action (sampled according to the logits),
        #2.log_prob of the action,
        #3.Entropy,
        #4.Value from the critic
        logits= self.actor(x)
        dist=Categorical(logits=logits)
        actions=dist.sample()
        logPs=dist.log_prob(actions)
        entropies=dist.entropy()
        value=self.get_value(x)

        return actions, logPs, entropies, value


In [70]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = torch_deterministic

device = torch.device("cuda" if torch.cuda.is_available() and cuda else "cpu")


In [71]:
#Make the vectorized environments, use the helper function that we have declared above
envs=gym.vector.SyncVectorEnv([make_env(gym_id, seed+i, i) for i in range(num_envs)])


In [72]:
agent = Agent(envs).to(device)
optimizer = optim.Adam(agent.parameters(), lr=learning_rate, eps=1e-5)

# ALGO Logic: Storage setup
obs = torch.zeros((num_steps, num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((num_steps, num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((num_steps, num_envs)).to(device)
rewards = torch.zeros((num_steps, num_envs)).to(device)
dones = torch.zeros((num_steps, num_envs)).to(device)
values = torch.zeros((num_steps, num_envs)).to(device)

In [73]:
# Start the game
global_step = 0
start_time = time.time()
next_obs, info = envs.reset()
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(num_envs).to(device)
num_updates = total_timesteps // batch_size

In [75]:
#This is the main training loop where we collect the experience ,
#calculate the advantages, ratio , the total loss and learn the policy

for update in range(1, num_updates + 1):

    # Annealing the rate if instructed to do so.
    if anneal_lr:
        # Your code here
        f=1.0-(update-1.0)/num_updates
        lrnow=f*learning_rate
        optimizer.param_groups[0]["lr"]=lrnow

        pass

    for step in range(0, num_steps):
        global_step += 1 * num_envs  # We are taking a step in each environment
        obs[step] = next_obs
        dones[step] = next_done

        # ALGO LOGIC: action logic
        with torch.no_grad():
            #Get the action , logprob , _ , value from the agent.
            action, logprob, _, value = agent.get_action_and_value(next_obs)

            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        # TRY NOT TO MODIFY: execute the game and log data.
        next_obs, reward, done,truncated, info = envs.step(action.cpu().numpy())
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(done).to(device)

        for item in info:
            if item == "final_info" and info[item][0] is not None:
                print(f"global_step={global_step}, episodic_return={info[item][0]['episode']['r']}")
                break

    # bootstrap value if not done
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        if gae:
          advantages=torch.zeros_like(rewards).to(device)
          lastgaelam=0
          for t in reversed(range(num_steps)):
            if t== num_steps-1:
              nextnonterminal=1-next_done
              nextvalues=next_value
            else:
              nextnonterminal=1-dones[t+1]
              nextvalues=values[t+1]
            delta=rewards[t]*gamma*nextvalues*nextnonterminal-values[t]
            advantages[t]=lastgaelam=delta+gamma*gae_lambda*nextnonterminal*lastgaelam

          returns = advantages + values  #(yes official implementation of ppo calculates it this way)
        else:
          returns=torch.zero_like(rewards).to(device)
          for t in  reversed(range(num_steps)):
            if t== num_steps-1:
              nextnonterminal=1-next_done
              next_return=next_value
            else:
              nextnonterminal=1-dones[t+1]
              next_return=returns[t+1]
            returns[t]=rewards[t]+gamma*nextnonterminal*next_return

            advantages = returns - values

    # flatten the batch
    b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    # Optimizing the policy and value network
    b_inds = np.arange(batch_size)
    clipfracs = []
    for epoch in range(update_epochs):
        #Get a random sample of batch_size
        np.random.shuffle(b_inds)
        for start in range(0, batch_size, minibatch_size):
            end = start + minibatch_size
            mb_inds = b_inds[start:end]

            #Your code here
            #Calculate the ratio
            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions.long()[mb_inds])
            logratio = newlogprob-b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                # calculate approx_kl http://joschu.net/blog/kl-approx.html
                # Refer the blog for calculating kl in a simpler way
                old_approx_kl = (-logratio).mean()
                approx_kl =((ratio-1)-logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > clip_coef).float().mean().item()]

            mb_advantages = b_advantages[mb_inds]
            if norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss (Calculate the policy loss pg_loss)
            # Your code here
            pg_loss1=-mb_advantages*ratio
            pg_loss2=-mb_advantages*torch.clamp(ratio, 1-clip_coef, 1+clip_coef)
            pg_loss=torch.max(pg_loss1, pg_loss2).mean()


            # Value loss v_loss
            newvalue = newvalue.view(-1)
            if clip_vloss:
                v_loss_unclipped=(newvalue-b_returns[mb_inds])**2
                v_clipped=b_values[mb_inds]+torch.clamp(
                    newvalue - b_values[mb_inds],
                    clip_coef,
                    clip_coef,
                )
                v_loss_clipped=(v_clipped-b_returns[mb_inds])**2
                v_loss_max=torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss=0.5*v_loss_max.mean()
            else:
                v_loss= 0.5*((newvalue - b_returns[mb_inds])**2).mean()
            # Entropy loss
            entropy_loss = entropy.mean()
            # Total loss
            loss = pg_loss - ent_coef * entropy_loss + v_loss * vf_coef

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), max_grad_norm)
            optimizer.step()

        if target_kl is not None:
            if approx_kl > target_kl:
                break

    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y


envs.close()

global_step=1584, episodic_return=[17.]
global_step=1636, episodic_return=[13.]
global_step=1748, episodic_return=[28.]
global_step=1832, episodic_return=[21.]
global_step=1896, episodic_return=[16.]
global_step=2012, episodic_return=[29.]
global_step=2160, episodic_return=[37.]
global_step=2208, episodic_return=[12.]
global_step=2312, episodic_return=[26.]
global_step=2396, episodic_return=[21.]
global_step=2464, episodic_return=[17.]
global_step=2532, episodic_return=[17.]
global_step=2636, episodic_return=[26.]
global_step=2696, episodic_return=[15.]
global_step=2824, episodic_return=[32.]
global_step=2888, episodic_return=[16.]
global_step=2968, episodic_return=[20.]
global_step=3020, episodic_return=[13.]


# Experiments and Plots
<a id="experiments"></a>

Run the DDPG, TD3, PPO on Pendulum, Hopper and Half Cheetah environment respectively.

Plot the following for each of the environment separately. Note based on different hyper-parameters and strategies you use, you can have multiple plots for each of the below.

As you are aware from your past experience, single run of the agent over the environment results in plots that have lot of variance and look very noisy. One way to overcome this is to create several different instances of the environment using different seeds and then average out the results across these and plot these. For all the plots below, you this strategy. You need to run 5 different instances of the environment for each agent. As you have seen in the lecture slides, we plot the maximum and minimum values around the mean in the plots, so this gives us the shaded plot with the mean curve in the between. In this assignment, you are required to do the same. Generate plots with envelop between maximum and minimum value
For each of the quantity of interest, plot each of the agent within the same plot using different colors for the envelop. Choose colors such that that there is clear contrast between the plots corresponding to different agents.

1. Plot mean train rewards vs episodes
2. Plot mean evaluation rewards vs episodes
3. Plot total steps vs episode
4. Plot train time vs episode
5. Plot wall clock time vs episode
6. Based on plots what are your observations about DDPG and TD3, compare the two algorithms.
7. What is the advatage of PPO over DDPG or TD3?